# Machine Condition Classification
**Multiclass classification: 0 = Normal, 1 = Thermal Failure, 2 = Mechanical Failure**

A machine-learning pipeline for predicting machine condition from sensor data, using only **Logistic Regression, Decision Tree and Random Forest**, combined in a weighted NNLS blend.

**Contents**
1. Introduction
2. Dataset overview
3. Exploratory data analysis
4. Preprocessing (missing values, encoding)
5. Feature engineering
6. Validation strategy and supervised models
7. Weighted ensemble (blend)
8. Per-class evaluation
9. Prediction generation
10. Key insights, limitations and conclusion

**Required files** in the same folder as this notebook: `student_dataset.csv`, `test.csv`
**Install:** `pip install pandas numpy scipy scikit-learn matplotlib seaborn`

## 1. Introduction
The goal is to build a system that predicts the operating condition of a machine unit. The input is machine type and sensor measurements. The classification can support condition monitoring and early fault detection.

The project covers EDA, preprocessing, feature engineering, supervised model experiments, performance comparison, cross-validation, and weighted ensemble prediction.

## 2. Dataset Overview
| Item | Description |
|---|---|
| Training data | 16,000 labelled machine units (`student_dataset.csv`) |
| Test data | 4,000 units requiring predictions (`test.csv`) |
| Target | `condition`: 0 = Normal; 1 = Thermal Failure; 2 = Mechanical Failure |
| Input features | `unit_type` (S, M, H) and ten numeric sensor readings |
| Sensors | ambient_temp, process_temp, coolant_flow, shaft_speed, drive_torque, power_load, component_wear, thermal_stress, mechanical_load, vibration_index |
| Missing values | Sparse, about 22–74 per feature column in training data |

The target is about 50% Normal, 25% Thermal Failure, 25% Mechanical Failure. Because Normal is larger than either failure class, accuracy is reported together with macro-F1.

In [ ]:
import os, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy.optimize import nnls
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PowerTransformer, PolynomialFeatures, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# ---------------- Configuration ----------------
DATA_DIR = "."          # folder containing student_dataset.csv and test.csv
OUT_DIR  = "./output"   # where results are written
BLEND    = 4            # 4 (default, selected final model) or 8 members

IN, OUT = DATA_DIR.rstrip("/") + "/", OUT_DIR.rstrip("/") + "/"
PLOTS = OUT + "eda_plots/"
os.makedirs(PLOTS, exist_ok=True)
sns.set_theme(style="whitegrid")

In [ ]:
tr = pd.read_csv(IN + "student_dataset.csv")
te = pd.read_csv(IN + "test.csv")
TARGET = "condition"
NUM = [c for c in te.columns if c not in ("unit_id", "unit_type")]
n, y = len(tr), tr[TARGET].values

print("Train", tr.shape, "| Test", te.shape)
print("\nClass distribution:\n", tr[TARGET].value_counts(normalize=True).sort_index())
print("\nMissing values (train):\n", tr.isna().sum()[lambda s: s > 0])
tr.head()

## 3. Exploratory Data Analysis
EDA examined the target distribution, missing values per column, sensor distributions, class-specific boxplots, sensor correlations, and unit type versus condition. Findings:
- Target is about 50% / 25% / 25%.
- Missing values are rare and scattered, so rows were retained rather than discarded.
- Sensor readings are related (e.g. torque, shaft speed and power load are physically linked).
- Unit types S, M, H appear 7,223, 5,537 and 3,218 times in training.
- These relationships motivated interaction and residual features.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x=TARGET, data=tr, ax=ax[0]); ax[0].set_title("Target distribution")
m_ = tr.isna().sum().sort_values(ascending=False)
m_[m_ > 0].plot.barh(ax=ax[1], color="tomato"); ax[1].set_title("Missing values per column")
plt.tight_layout(); plt.savefig(PLOTS + "01_target_missing.png", dpi=120); plt.show()

In [ ]:
tr[NUM].hist(bins=40, figsize=(14, 9), color="steelblue")
plt.tight_layout(); plt.savefig(PLOTS + "02_histograms.png", dpi=120); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for a, c in zip(axes.ravel(), NUM):
    sns.boxplot(x=TARGET, y=c, data=tr, ax=a, showfliers=False)
plt.tight_layout(); plt.savefig(PLOTS + "03_boxplots_by_class.png", dpi=120); plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(tr[NUM + [TARGET]].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.tight_layout(); plt.savefig(PLOTS + "04_correlation.png", dpi=120); plt.show()

In [ ]:
print(tr.unit_type.value_counts())
pd.crosstab(tr.unit_type, tr[TARGET], normalize="index").plot(kind="bar", stacked=True, figsize=(6, 4))
plt.tight_layout(); plt.savefig(PLOTS + "05_unit_type.png", dpi=120); plt.show()

## 4. Preprocessing
**Missing values.** Iterative imputation predicts each sensor from the others. Missing-value indicator columns are kept so models can use which values were originally absent. The imputer is fitted on combined train + test **features only** (no labels). This is a transductive choice; change it to training-only fitting if your evaluation protocol requires.

**Encoding.** `unit_type`: S = 0, M = 1, H = 2.

In [ ]:
allx = pd.concat([tr.drop(columns=TARGET), te], ignore_index=True)
allx["unit_type"] = allx.unit_type.map({"S": 0, "M": 1, "H": 2})
flags = allx[NUM].isna().astype(int).add_suffix("_nan")          # missing-value indicators

Z = pd.DataFrame(IterativeImputer(max_iter=10, random_state=0).fit_transform(allx[NUM + ["unit_type"]]),
                 columns=NUM + ["unit_type"])
Z["unit_type"] = Z.unit_type.round().clip(0, 2)
print("Remaining NaN after imputation:", Z.isna().sum().sum())

## 5. Feature Engineering
- **Physical:** `temp_delta = process_temp − ambient_temp`; estimated power = torque × speed / 9549; `power_gap = power_load − estimated power`.
- **Interaction / ratio:** wear × load, wear × torque, wear × vibration; heat per coolant flow; load per coolant flow.
- **Residuals:** for eight sensors, observed value minus value predicted (linear regression) from the other sensors, highlighting deviations from expected relationships.
- **Missingness indicators:** binary flags for originally missing sensor values.

In [ ]:
def fe(Z):
    d = Z.copy()
    d["temp_delta"] = d.process_temp - d.ambient_temp
    d["wear_x_load"] = d.component_wear * d.mechanical_load
    d["torque_x_speed"] = d.drive_torque * d.shaft_speed / 9549
    d["wear_x_torque"] = d.component_wear * d.drive_torque
    d["heat_per_flow"] = d.process_temp / (d.coolant_flow + 1)
    d["wear_x_vib"] = d.component_wear * d.vibration_index
    d["load_per_flow"] = d.mechanical_load / (d.coolant_flow + 1)
    d["power_gap"] = d.power_load - d.torque_x_speed
    for c in ["process_temp", "thermal_stress", "vibration_index", "mechanical_load",
              "component_wear", "coolant_flow", "drive_torque", "shaft_speed"]:
        o = [k for k in NUM if k != c]
        d[c + "_res"] = Z[c] - LinearRegression().fit(Z[o], Z[c]).predict(Z[o])  # deviation from expected relationship
    return d

F = fe(Z)
RAW  = NUM + ["unit_type"]                                   # 11 original columns
CORE = list(F.columns)                                       # original + engineered
SEL  = RAW + ["temp_delta", "wear_x_load", "power_gap", "heat_per_flow"]
X_all = pd.concat([F, flags], axis=1)                        # + missing-value flags
Xtr, Xte = X_all.iloc[:n].reset_index(drop=True), X_all.iloc[n:].reset_index(drop=True)
print("Feature matrix:", Xtr.shape, Xte.shape)

## 6. Validation Strategy and Supervised Models
Models are compared with **stratified 5-fold cross-validation**. Out-of-fold (OOF) probabilities are produced for training rows, and test probabilities are averaged across the five fold models. Accuracy and macro-F1 are both reported.

Members tested: plain Logistic Regression (baseline), polynomial (degree 2 and 3) Logistic Regression, Decision Tree (depth 8), and Random Forest variants.

In [ ]:
poly = lambda deg, C: make_pipeline(PowerTransformer(), PolynomialFeatures(deg, include_bias=False),
                                    StandardScaler(), LogisticRegression(C=C, max_iter=4000))
ALL = list(Xtr.columns)
# name: (model factory, columns used)
all_members = {
    "LogReg poly3 (raw, C=0.1)":     (lambda: poly(3, 0.1),  RAW),
    "LogReg poly3 (raw, C=0.05)":    (lambda: poly(3, 0.05), RAW),
    "LogReg poly3 (selected)":       (lambda: poly(3, 0.1),  SEL),
    "LogReg poly2 (all engineered)": (lambda: poly(2, 0.3),  CORE),
    "LogReg plain":                  (lambda: make_pipeline(PowerTransformer(), LogisticRegression(C=1, max_iter=3000)), CORE),
    "DecisionTree depth 8":          (lambda: DecisionTreeClassifier(max_depth=8, min_samples_leaf=10, random_state=0), ALL),
    "RandomForest leaf2":            (lambda: RandomForestClassifier(300, min_samples_leaf=2, n_jobs=-1, random_state=0), ALL),
    "RandomForest 50% cols":         (lambda: RandomForestClassifier(300, max_features=0.5, min_samples_leaf=1, n_jobs=-1, random_state=1), ALL),
}
FOUR = ["LogReg poly3 (raw, C=0.05)", "LogReg poly2 (all engineered)", "DecisionTree depth 8", "RandomForest 50% cols"]
members = {k: v for k, v in all_members.items() if BLEND == 8 or k in FOUR}
suffix = f"_{BLEND}_model_blend"

In [ ]:
folds = list(StratifiedKFold(5, shuffle=True, random_state=1).split(Xtr, y))
oofs, tests, rows = {}, {}, []
for name, (mk, cols) in members.items():
    oof, tp = np.zeros((n, 3)), np.zeros((len(Xte), 3))
    for a, b in folds:
        m = mk().fit(Xtr.loc[a, cols], y[a])
        oof[b] = m.predict_proba(Xtr.loc[b, cols])
        tp += m.predict_proba(Xte[cols]) / len(folds)
    oofs[name], tests[name] = oof, tp
    p = oof.argmax(1)
    rows.append(dict(model=name, cv_accuracy=accuracy_score(y, p), cv_macro_f1=f1_score(y, p, average="macro")))
    print(f"{name:32s} acc={rows[-1]['cv_accuracy']:.4f}  macroF1={rows[-1]['cv_macro_f1']:.4f}", flush=True)

### Reference results from the report (all 8 configurations)
| Model configuration | CV accuracy | CV macro-F1 |
|---|---|---|
| Logistic Regression, degree 3 (raw, C=0.1) | 0.8879 | 0.8782 |
| Logistic Regression, degree 3 (raw, C=0.05) | 0.8879 | 0.8783 |
| Logistic Regression, degree 3 (selected columns) | 0.8874 | 0.8773 |
| Logistic Regression, degree 2 (engineered) | 0.8861 | 0.8765 |
| Logistic Regression, plain linear | 0.8521 | 0.8370 |
| Decision Tree, depth 8 | 0.8437 | 0.8239 |
| Random Forest, min leaf 2 | 0.8813 | 0.8693 |
| Random Forest, 50% columns per split | 0.8819 | 0.8703 |

Degree-3 polynomial Logistic Regression was the strongest individual model; Random Forest was a close second; the Decision Tree was weakest. Set `BLEND = 8` in the configuration cell to reproduce all eight rows.

## 7. Weighted Ensemble (Final Model)
Blend weights are estimated by **non-negative least squares (NNLS)** on out-of-fold probabilities against one-hot targets, then normalised to sum to one.

Reported weights for the four-model blend: LogReg poly3 raw C=0.05 → 0.545, LogReg poly2 engineered → 0.268, Random Forest 50% cols → 0.161, Decision Tree depth 8 → 0.025. Reported CV accuracy **0.8904**, macro-F1 **0.8809**. The eight-member blend gave the same metrics, so the simpler four-model blend was retained.

In [ ]:
names = list(members)
M = np.column_stack([oofs[k].ravel() for k in names])       # (n*3, members)
T = np.stack([tests[k] for k in names], axis=-1)            # (n_test, 3, members)
w, _ = nnls(M, np.eye(3)[y].ravel()); w = w / w.sum()
print("Blend weights:", {k: round(v, 3) for k, v in zip(names, w)})

P_oof = np.stack([oofs[k] for k in names], axis=-1) @ w
p_oof = P_oof.argmax(1)
p_avg = np.mean([oofs[k] for k in names], 0).argmax(1)
print(f"Simple average  acc={accuracy_score(y, p_avg):.4f} macroF1={f1_score(y, p_avg, average='macro'):.4f}")
print(f"Weighted blend  acc={accuracy_score(y, p_oof):.4f} macroF1={f1_score(y, p_oof, average='macro'):.4f}")

rows.append(dict(model=f"BLEND ({BLEND} models, weighted)",
                 cv_accuracy=accuracy_score(y, p_oof), cv_macro_f1=f1_score(y, p_oof, average="macro")))
res = pd.DataFrame(rows)
res.to_csv(OUT + f"cv_results{suffix}.csv", index=False)
res

In [ ]:
res.set_index("model").plot.barh(figsize=(10, 6), xlim=(0.7, 0.92))
plt.title("5-fold CV: members vs blend"); plt.tight_layout()
plt.savefig(PLOTS + "06_model_comparison.png", dpi=120); plt.show()

## 8. Per-Class Evaluation
Reported (4-model blend): Normal P/R/F1 = 0.889 / 0.943 / 0.916; Thermal = 0.890 / 0.841 / 0.865; Mechanical = 0.893 / 0.834 / 0.863.
The largest errors are failure cases predicted as Normal (453 thermal and 487 mechanical in the reported OOF matrix). Thermal and mechanical failures are rarely confused with each other.

In [ ]:
print(classification_report(y, p_oof, target_names=["Normal", "Thermal", "Mechanical"], digits=3))
cm = confusion_matrix(y, p_oof)
print("Confusion matrix:\n", cm)
lbl = ["Normal", "Thermal", "Mechanical"]
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=lbl, yticklabels=lbl)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Out-of-fold confusion matrix")
plt.tight_layout(); plt.savefig(PLOTS + "07_confusion_matrix.png", dpi=120); plt.show()

## 9. Prediction Generation
The blend produces one condition label per test unit (4,000 rows). Output: `predictions_4_model_blend.csv` (selected) or 'predictions_8_model_blend.csv'   with columns `unit_id`, `condition`.

In [ ]:
pred = (T @ w).argmax(1)
sub = pd.DataFrame({"unit_id": te.unit_id, TARGET: pred})
sub.to_csv(OUT + f"predictions{suffix}.csv", index=False)
print("Saved", OUT + f"predictions{suffix}.csv", sub.shape)
print(sub[TARGET].value_counts(normalize=True).sort_index())
sub.head()

## 10. Key Insights, Limitations and Conclusion
**Insights**
- Polynomial Logistic Regression was the strongest individual model; degree-3 clearly beat plain linear (0.8879 vs 0.8521 accuracy).
- Random Forest was a close second and contributes complementary predictions.
- The standalone Decision Tree was weakest and received a small blend weight; it is retained so all three permitted model families are represented.
- Blending lifted accuracy from ~0.8879 to ~0.8904; the 8-member blend matched the 4-member blend.
- Failure recall (≈0.83–0.84) is lower than Normal recall (0.943); catching every failure is the main remaining challenge.

**Limitations**
- CV accuracy is around 0.89; further tuning gave only small gains.
- Some thermal and mechanical failures are missed and labelled Normal.
- Imputer and residual features use train + test features (no labels); this transductive choice may not suit every evaluation protocol.
- CV estimates may differ from performance on future or hidden data.

**Not performed:** no 20% dataset-size experiment, threshold-optimisation experiment or unsupervised clustering experiment is part of this project.

**Conclusion.** The project delivers a complete three-class workflow: EDA, iterative imputation with missingness flags, physical/interaction/residual features, stratified 5-fold CV, and an NNLS-weighted blend of Logistic Regression, Decision Tree and Random Forest. The final four-model ensemble reaches CV accuracy 0.8904 and macro-F1 0.8809. Improving failure recall is the main opportunity for future work.

**Deliverables produced by this notebook** (in `OUT_DIR`): `predictions_4_model_blend.csv`, `cv_results_4_model_blend.csv`, `eda_plots/`.